# **SpaceX Falcon 9 First Stage Landing Prediction**
## **Lab 2: Web Scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia**

In this lab, we perform web scraping using `BeautifulSoup` and `requests` to collect Falcon 9 historical launch records from a Wikipedia snapshot (`List of Falcon 9 and Falcon Heavy launches`). We extract table column headers, parse HTML table rows across launch tables, clean HTML tags and footnotes, structure the data into a Pandas DataFrame, and export the dataset to `spacex_web_scraped.csv`.

### **Objectives**
* Request the static Falcon 9 Launch Wikipedia page using HTTP `GET`.
* Parse the HTML content into a `BeautifulSoup` object (`html.parser`).
* Extract column names from the `wikitable plainrowheaders collapsible` launch tables.
* Parse all launch rows and clean embedded HTML tags/footnotes using helper functions.
* Export the resulting structured DataFrame to `spacex_web_scraped.csv`.

### **1. Import Required Libraries & Helper Functions**

In [1]:
import sys
import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

print("Web scraping libraries imported successfully.")

Web scraping libraries imported successfully.


In [2]:
# Helper functions to clean HTML table cells from Wikipedia
def date_time(table_cells):
    """Returns the date and time from the HTML table cell"""
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """Returns the booster version from the HTML table cell"""
    out = ''.join([booster_version for i, booster_version in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out

def landing_status(table_cells):
    """Returns the landing status from the HTML table cell"""
    out = [i for i in table_cells.strings][0]
    return out

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass = mass[0:mass.find("kg") + 2]
    else:
        new_mass = 0
    return new_mass

def extract_column_from_header(row):
    """Extracts and cleans column header names from <th> elements"""
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    column_name = ' '.join(row.contents)
    if not (column_name.strip().isdigit()):
        column_name = column_name.strip()
        return column_name

print("Helper parsing functions defined.")

Helper parsing functions defined.


### **2. Request the Falcon 9 Launch Wiki Page & Create BeautifulSoup Object**

In [3]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

response = requests.get(static_url)
print("HTTP Status Code:", response.status_code)

soup = BeautifulSoup(response.text, 'html.parser')
print("Page Title:", soup.title.string)

HTTP Status Code: 200
Page Title: List of Falcon 9 and Falcon Heavy launches - Wikipedia


### **3. Extract All Column/Variable Names from the HTML Table Header**

In [4]:
html_tables = soup.find_all('table')
print("Total HTML tables found on page:", len(html_tables))

# Target the third table containing Falcon 9 launch records
first_launch_table = html_tables[2]

column_names = []
for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)

print("Extracted Column Names:")
print(column_names)

Total HTML tables found on page: 25
Extracted Column Names:
['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


### **4. Create a DataFrame by Parsing the Launch HTML Tables**

In [5]:
launch_dict = dict.fromkeys(column_names)
del launch_dict['Date and time ( )']

# Initialize dictionary lists
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []

extracted_row = 0
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    for rows in table.find_all("tr"):
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False
        row = rows.find_all('td')
        if flag:
            extracted_row += 1
            launch_dict['Flight No.'].append(flight_number)
            
            datatimelist = date_time(row[0])
            launch_dict['Date'].append(datatimelist[0].strip(','))
            launch_dict['Time'].append(datatimelist[1])
            
            bv = booster_version(row[1])
            if not bv:
                bv = row[1].a.string
            launch_dict['Version Booster'].append(bv)
            
            launch_dict['Launch site'].append(row[2].a.string)
            launch_dict['Payload'].append(row[3].a.string)
            launch_dict['Payload mass'].append(get_mass(row[4]))
            launch_dict['Orbit'].append(row[5].a.string)
            
            customer = row[6].a.string if row[6].a else row[6].text.strip()
            launch_dict['Customer'].append(customer)
            
            launch_dict['Launch outcome'].append(list(row[7].strings)[0].strip())
            launch_dict['Booster landing'].append(landing_status(row[8]).strip())

print(f"Total launch rows extracted: {extracted_row}")

Total launch rows extracted: 121


In [6]:
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})
print("Scraped DataFrame Shape:", df.shape)
df.head(5)

Scraped DataFrame Shape: (121, 11)


  Flight No. Launch site                               Payload Payload mass Orbit Customer Launch outcome Version Booster Booster landing        Date   Time
0          1       CCAFS  Dragon Spacecraft Qualification Unit            0   LEO   SpaceX        Success  F9 v1.0B0003.1         Failure  4 June 2010  18:45
1          2       CCAFS                                Dragon            0   LEO     NASA        Success  F9 v1.0B0004.1         Failure  8 December 2010  15:43
2          3       CCAFS                                Dragon       525 kg   LEO     NASA        Success  F9 v1.0B0005.1    No attempt  22 May 2012  07:44
3          4       CCAFS                          SpaceX CRS-1     4,700 kg   LEO     NASA        Success  F9 v1.0B0006.1      No attempt  8 October 2012  00:35
4          5       CCAFS                          SpaceX CRS-2     4,877 kg   LEO     NASA        Success  F9 v1.0B0007.1    No attempt  1 March 2013  15:10

### **5. Export Scraped Data to CSV**

In [7]:
df.to_csv('spacex_web_scraped.csv', index=False)
print("Successfully exported spacex_web_scraped.csv (121 rows, 11 columns).")

Successfully exported spacex_web_scraped.csv (121 rows, 11 columns).
